<a href="https://colab.research.google.com/github/krishnasivaprasadm-jpg/cardiovascular_CA-SAE-AFB-a/blob/main/crossdomain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# 🔥 FINAL ALL-IN-ONE PIPELINE (CARDIO DATASET)
# ============================================================

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, matthews_corrcoef
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.utils.class_weight import compute_class_weight
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings("ignore")

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# ============================================================
# 📌 LOAD + CLEAN DATA
# ============================================================

df = pd.read_csv("cardio_train.csv", sep=";")

# Drop useless column
if "id" in df.columns:
    df = df.drop(columns=["id"])

# Convert age (days → years)
df["age"] = df["age"] / 365

# Remove outliers (VERY IMPORTANT)
df = df[(df["ap_hi"] < 250) & (df["ap_lo"] < 200)]
df = df[(df["ap_hi"] > 50) & (df["ap_lo"] > 30)]

target_col = "cardio"

# One-hot encoding
df = pd.get_dummies(df, drop_first=True)

X_full = df.drop(target_col, axis=1).values
y_full = df[target_col].values

# ============================================================
# 📌 MODEL: CA-SAE-AFB
# ============================================================

def build_ca_sae_afb(input_dim):

    inputs = layers.Input(shape=(input_dim,))

    x = layers.Dense(128, activation='relu',
                     activity_regularizer=tf.keras.regularizers.l1(1e-5))(inputs)
    x = layers.Dropout(0.4)(x)

    z = layers.Dense(64, activation='relu')(x)

    weights = layers.Dense(64, activation='sigmoid')(z)
    z_boosted = layers.Multiply()([z, weights])

    x_hat = layers.Dense(input_dim, activation='linear')(z_boosted)
    y_out = layers.Dense(1, activation='sigmoid')(z_boosted)

    model = models.Model(inputs, [x_hat, y_out])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(0.0003),
        loss=['mse', 'binary_crossentropy'],
        loss_weights=[0.3, 1.0]
    )

    encoder = models.Model(inputs, z_boosted)

    return model, encoder

# ============================================================
# 📌 MAIN TRAINING (ONLY SEED = 9)
# ============================================================

print("🚀 Running FINAL pipeline for SEED = 9...\n")

seed = 9  # ✅ ONLY CHANGE APPLIED

# -----------------------------
# Split
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X_full, y_full,
    test_size=0.2,
    stratify=y_full,
    random_state=seed
)

# -----------------------------
# Interaction Features
# -----------------------------
poly = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
X_train = poly.fit_transform(X_train)
X_test = poly.transform(X_test)

# -----------------------------
# Scaling
# -----------------------------
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# -----------------------------
# Class Weights
# -----------------------------
class_weights = compute_class_weight(
    'balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weights = dict(enumerate(class_weights))

# -----------------------------
# CA-SAE-AFB Training
# -----------------------------
cae, encoder = build_ca_sae_afb(X_train.shape[1])

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='loss',
    patience=8,
    restore_best_weights=True
)

cae.fit(
    X_train, [X_train, y_train],
    epochs=100,
    batch_size=64,
    verbose=0,
    callbacks=[early_stop]
)

# Latent Features
Z_train = encoder.predict(X_train, verbose=0)
Z_test = encoder.predict(X_test, verbose=0)

# -----------------------------
# MODEL 1 (Logistic)
# -----------------------------
clf_nn = LogisticRegression(max_iter=2000)
clf_nn.fit(Z_train, y_train)

P1_train = clf_nn.predict_proba(Z_train)[:,1]
P1_test = clf_nn.predict_proba(Z_test)[:,1]

# -----------------------------
# MODEL 2 (XGBoost)
# -----------------------------
xgb = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=3,
    eval_metric='logloss',
    random_state=seed
)

xgb.fit(X_train, y_train)

P2_train = xgb.predict_proba(X_train)[:,1]
P2_test = xgb.predict_proba(X_test)[:,1]

# -----------------------------
# META MODEL
# -----------------------------
meta_X_train = np.column_stack((P1_train, P2_train))
meta_X_test = np.column_stack((P1_test, P2_test))

meta = GradientBoostingClassifier(n_estimators=100)
meta.fit(meta_X_train, y_train)

P_meta_test = meta.predict_proba(meta_X_test)[:,1]

# -----------------------------
# FINAL PREDICTION
# -----------------------------
y_pred = (P_meta_test > 0.5).astype(int)

acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, P_meta_test)
f1 = f1_score(y_test, y_pred)
mcc = matthews_corrcoef(y_test, y_pred)

# ============================================================
# 📌 FINAL RESULTS
# ============================================================

print("\n==============================")
print(" FINAL RESULT (SEED = 9)")
print("==============================")
print("Accuracy :", acc)
print("ROC-AUC  :", auc)
print("F1 Score :", f1)
print("MCC      :", mcc)

🚀 Running FINAL pipeline for SEED = 9...


 FINAL RESULT (SEED = 9)
Accuracy : 0.7291893856779353
ROC-AUC  : 0.7986957975885671
F1 Score : 0.7142747564623763
MCC      : 0.4594786041483457


In [ ]:
# ============================================================
# FINAL MODEL — 30 SEEDS + PER-SEED OUTPUT
# ============================================================

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, matthews_corrcoef
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings("ignore")

np.random.seed(42)
tf.random.set_seed(42)

# ============================================================
# LOAD DATA
# ============================================================

df = pd.read_csv("cardio_train.csv", sep=";")

if "id" in df.columns:
    df = df.drop("id", axis=1)

df["age"] = df["age"] / 365.0

# Clean BP
df = df[(df["ap_hi"] < 250) & (df["ap_lo"] < 200)]
df = df[(df["ap_hi"] > 50) & (df["ap_lo"] > 30)]

target_col = "cardio"

df = pd.get_dummies(df, drop_first=True)

X_full = df.drop(target_col, axis=1).values
y_full = df[target_col].values

print("Dataset shape:", X_full.shape)
print("Class distribution:", np.bincount(y_full))

# ============================================================
# MODEL
# ============================================================

def build_ca_sae_afb(input_dim):

    inputs = layers.Input(shape=(input_dim,))

    x = layers.Dense(128, activation='relu',
                     activity_regularizer=tf.keras.regularizers.l1(1e-4))(inputs)
    x = layers.Dropout(0.3)(x)

    z = layers.Dense(64, activation='relu')(x)

    weights = layers.Dense(64, activation='sigmoid')(z)
    z = layers.Multiply()([z, weights])

    x_hat = layers.Dense(input_dim)(z)
    y_out = layers.Dense(1, activation='sigmoid')(z)

    model = models.Model(inputs, [x_hat, y_out])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(0.0005),
        loss=['mse', 'binary_crossentropy'],
        loss_weights=[0.5, 1.0]
    )

    encoder = models.Model(inputs, z)

    return model, encoder

# ============================================================
# MULTI-SEED RUN
# ============================================================

all_acc, all_auc, all_f1, all_mcc = [], [], [], []

print("\n================ PER-SEED RESULTS ================\n")

for seed in range(30):

    # SPLIT
    X_train, X_test, y_train, y_test = train_test_split(
        X_full, y_full,
        test_size=0.2,
        stratify=y_full,
        random_state=seed
    )

    # INTERACTION
    poly = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
    X_train = poly.fit_transform(X_train)
    X_test = poly.transform(X_test)

    # SCALING
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    # CA-SAE-AFB
    cae, encoder = build_ca_sae_afb(X_train.shape[1])

    early_stop = tf.keras.callbacks.EarlyStopping(
        monitor='loss',
        patience=10,
        restore_best_weights=True
    )

    cae.fit(
        X_train, [X_train, y_train],
        epochs=100,
        batch_size=64,
        verbose=0,
        callbacks=[early_stop]
    )

    Z_train = encoder.predict(X_train, verbose=0)
    Z_test = encoder.predict(X_test, verbose=0)

    # BASE 1
    lr = LogisticRegression(max_iter=3000)
    lr.fit(Z_train, y_train)

    P1_train = lr.predict_proba(Z_train)[:,1]
    P1_test = lr.predict_proba(Z_test)[:,1]

    # BASE 2
    xgb = XGBClassifier(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.03,
        subsample=0.9,
        colsample_bytree=0.9,
        reg_lambda=2,
        reg_alpha=0.5,
        eval_metric='logloss',
        random_state=seed
    )

    xgb.fit(X_train, y_train)

    P2_train = xgb.predict_proba(X_train)[:,1]
    P2_test = xgb.predict_proba(X_test)[:,1]

    # META
    meta_X_train = np.column_stack((P1_train, P2_train))
    meta_X_test = np.column_stack((P1_test, P2_test))

    meta = GradientBoostingClassifier(n_estimators=200)
    meta.fit(meta_X_train, y_train)

    P_meta_train = meta.predict_proba(meta_X_train)[:,1]
    P_meta_test = meta.predict_proba(meta_X_test)[:,1]

    # THRESHOLD
    best_thresh = 0.5
    best_f1 = 0

    for t in np.arange(0.3, 0.7, 0.002):
        preds = (P_meta_train > t).astype(int)
        f1 = f1_score(y_train, preds)
        if f1 > best_f1:
            best_f1 = f1
            best_thresh = t

    # TWO-STAGE
    T_low = best_thresh - 0.05
    T_high = best_thresh + 0.05

    y_pred = []

    for p in P_meta_test:
        if p >= T_high:
            y_pred.append(1)
        elif p <= T_low:
            y_pred.append(0)
        else:
            y_pred.append(1 if p > best_thresh else 0)

    y_pred = np.array(y_pred)

    # METRICS
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, P_meta_test)
    f1 = f1_score(y_test, y_pred)
    mcc = matthews_corrcoef(y_test, y_pred)

    all_acc.append(acc)
    all_auc.append(auc)
    all_f1.append(f1)
    all_mcc.append(mcc)

    print(f"Seed {seed:02d} → "
          f"Acc: {acc:.4f}, "
          f"AUC: {auc:.4f}, "
          f"F1: {f1:.4f}, "
          f"MCC: {mcc:.4f}")

# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n================ FINAL RESULTS (30 SEEDS) ================")

print(f"Accuracy : {np.mean(all_acc):.4f} ± {np.std(all_acc):.4f}")
print(f"ROC-AUC  : {np.mean(all_auc):.4f} ± {np.std(all_auc):.4f}")
print(f"F1 Score : {np.mean(all_f1):.4f} ± {np.std(all_f1):.4f}")
print(f"MCC      : {np.mean(all_mcc):.4f} ± {np.std(all_mcc):.4f}")

Dataset shape: (68775, 11)
Class distribution: [34738 34037]

================ PER-SEED RESULTS ================

Seed 00 → Acc: 0.7081, AUC: 0.7803, F1: 0.7152, MCC: 0.4176
Seed 01 → Acc: 0.6998, AUC: 0.7705, F1: 0.7132, MCC: 0.4029
Seed 02 → Acc: 0.7011, AUC: 0.7742, F1: 0.7133, MCC: 0.4050
Seed 03 → Acc: 0.7125, AUC: 0.7800, F1: 0.7231, MCC: 0.4275
Seed 04 → Acc: 0.6979, AUC: 0.7734, F1: 0.7105, MCC: 0.3986
Seed 05 → Acc: 0.6987, AUC: 0.7733, F1: 0.7157, MCC: 0.4022
Seed 06 → Acc: 0.7065, AUC: 0.7783, F1: 0.7194, MCC: 0.4162
Seed 07 → Acc: 0.7000, AUC: 0.7730, F1: 0.7112, MCC: 0.4024
Seed 08 → Acc: 0.6998, AUC: 0.7779, F1: 0.7155, MCC: 0.4038
Seed 09 → Acc: 0.7029, AUC: 0.7771, F1: 0.7170, MCC: 0.4095
Seed 10 → Acc: 0.6980, AUC: 0.7709, F1: 0.7129, MCC: 0.3998
Seed 11 → Acc: 0.6993, AUC: 0.7730, F1: 0.7125, MCC: 0.4018
Seed 12 → Acc: 0.7117, AUC: 0.7836, F1: 0.7188, MCC: 0.4248
Seed 13 → Acc: 0.6987, AUC: 0.7730, F1: 0.7109, MCC: 0.4002
Seed 14 → Acc: 0.7144, AUC: 0.7841, F1: 0.7207

In [ ]:
# ============================================================
# MPML IMPLEMENTATION (FIXED + FULL RUNNABLE)
# ============================================================

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import StandardScaler
from sklearn.calibration import CalibratedClassifierCV

from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

# ============================================================
# SETTINGS
# ============================================================

SEED = 9
np.random.seed(SEED)

USE_EXPERT_GROUPS = True

# ============================================================
# LOAD DATA
# ============================================================

df = pd.read_csv("/content/Cardiovascular_Disease_Dataset.csv")   # your dataset

target_col = "target"  # change if needed

X = df.drop(columns=[target_col])
y = df[target_col]

print("Dataset shape:", X.shape)
print("Columns:", X.columns.tolist())

# ============================================================
# SAFE FEATURE HANDLING (IMPORTANT FIX)
# ============================================================

def validate_perspectives(perspectives, X):
    valid = []
    for p in perspectives:
        p_valid = [f for f in p if f in X.columns]
        if len(p_valid) > 0:
            valid.append(p_valid)
    return valid

# ============================================================
# 1. FEATURE GROUPING
# ============================================================

def mutual_info_grouping(X, y, n_groups=3):
    mi = mutual_info_classif(X, y, random_state=SEED)
    mi = (mi - mi.min()) / (mi.max() - mi.min() + 1e-9)

    km = KMeans(n_clusters=n_groups, random_state=SEED)
    labels = km.fit_predict(mi.reshape(-1, 1))

    groups = []
    for i in range(n_groups):
        groups.append(X.columns[labels == i].tolist())

    return groups


def correlation_grouping(X, n_groups=3):
    corr = np.abs(X.corr())
    dist = 1 - corr

    condensed = squareform(dist)
    Z = linkage(condensed, method='average')

    labels = fcluster(Z, n_groups, criterion='maxclust')

    groups = []
    for i in range(1, n_groups + 1):
        groups.append(X.columns[labels == i].tolist())

    return groups


def pca_grouping(X, n_groups=3):
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    X_t = X_scaled.T

    pca = PCA(n_components=3, random_state=SEED)
    X_pca = pca.fit_transform(X_t)

    km = KMeans(n_clusters=n_groups, random_state=SEED)
    labels = km.fit_predict(X_pca)

    groups = []
    for i in range(n_groups):
        groups.append(X.columns[labels == i].tolist())

    return groups


def model_importance_grouping(X, y, n_groups=3):
    rf = RandomForestClassifier(n_estimators=23, random_state=SEED)
    rf.fit(X, y)

    imp = rf.feature_importances_
    imp = (imp - imp.min()) / (imp.max() - imp.min() + 1e-9)

    km = KMeans(n_clusters=n_groups, random_state=SEED)
    labels = km.fit_predict(imp.reshape(-1, 1))

    groups = []
    for i in range(n_groups):
        groups.append(X.columns[labels == i].tolist())

    return groups


# ============================================================
# EXPERT GROUPING (AUTO SAFE)
# ============================================================

def expert_grouping(X):
    possible_groups = [
        ['age', 'gender'],
        ['ap_hi', 'ap_lo'],
        ['cholesterol', 'gluc'],
        ['smoke', 'alco', 'active'],
        ['height', 'weight']
    ]

    return validate_perspectives(possible_groups, X)


# ============================================================
# BUILD PERSPECTIVES
# ============================================================

def build_perspectives(X, y):
    perspectives = []

    perspectives += mutual_info_grouping(X, y)
    perspectives += correlation_grouping(X)
    perspectives += pca_grouping(X)
    perspectives += model_importance_grouping(X, y)

    if USE_EXPERT_GROUPS:
        perspectives += expert_grouping(X)

    perspectives = validate_perspectives(perspectives, X)

    return perspectives


# ============================================================
# MPML MODEL
# ============================================================

def train_mpml(X_train, y_train, perspectives):
    models = []

    for p in perspectives:
        model = DecisionTreeClassifier(random_state=SEED)
        model.fit(X_train[p], y_train)
        models.append((model, p))

    return models


def predict_mpml(models, X):
    probs = []

    for model, p in models:
        prob = model.predict_proba(X[p])[:, 1]
        probs.append(prob)

    final_prob = np.mean(probs, axis=0)
    preds = (final_prob >= 0.5).astype(int)

    return preds, final_prob


# ============================================================
# INTERPRETABILITY
# ============================================================

def compute_impact_scores(models, X_instance):
    _, base_prob = predict_mpml(models, X_instance)
    base_prob = base_prob[0]

    impacts = []

    for i in range(len(models)):
        reduced = models[:i] + models[i+1:]
        _, new_prob = predict_mpml(reduced, X_instance)
        new_prob = new_prob[0]

        impacts.append(base_prob - new_prob)

    return impacts


def compute_feature_impact(models, perspective, X_instance):
    _, base_prob = predict_mpml(models, X_instance)
    base_prob = base_prob[0]

    impacts = {}

    for f in perspective:
        X_mod = X_instance.copy()
        X_mod[f] = 0

        _, new_prob = predict_mpml(models, X_mod)
        new_prob = new_prob[0]

        impacts[f] = base_prob - new_prob

    return impacts


# ============================================================
# EVALUATION
# ============================================================

def evaluate(y_true, y_pred):
    return [
        accuracy_score(y_true, y_pred),
        precision_score(y_true, y_pred),
        recall_score(y_true, y_pred),
        f1_score(y_true, y_pred)
    ]


# ============================================================
# CROSS VALIDATION
# ============================================================

def run_mpml_cv(X, y):
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED)

    results = []

    for train_idx, test_idx in skf.split(X, y):
        X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
        y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]

        perspectives = build_perspectives(X_tr, y_tr)
        models = train_mpml(X_tr, y_tr, perspectives)

        preds, _ = predict_mpml(models, X_te)
        results.append(evaluate(y_te, preds))

    results = np.array(results)

    return results.mean(axis=0), results.std(axis=0)


# ============================================================
# BASELINES
# ============================================================

def run_baselines(X, y):
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED)

    models = {
        "RandomForest": RandomForestClassifier(n_estimators=23, random_state=SEED),
        "Bagging": BaggingClassifier(
            estimator=DecisionTreeClassifier(),
            n_estimators=16,
            random_state=SEED
        ),
        "Boosting": GradientBoostingClassifier(n_estimators=23, random_state=SEED)
    }

    results = {}

    for name, model in models.items():
        scores = []

        for tr, te in skf.split(X, y):
            model.fit(X.iloc[tr], y.iloc[tr])
            preds = model.predict(X.iloc[te])
            scores.append(evaluate(y.iloc[te], preds))

        scores = np.array(scores)
        results[name] = (scores.mean(axis=0), scores.std(axis=0))

    return results


# ============================================================
# RUN
# ============================================================

print("\n🚀 Running MPML...")

mpml_mean, mpml_std = run_mpml_cv(X, y)

print("\nMPML Results:")
print("Accuracy Precision Recall F1")
print(mpml_mean)
print(mpml_std)

print("\n🚀 Running Baselines...")

baseline = run_baselines(X, y)

for name, (mean, std) in baseline.items():
    print(f"\n{name}")
    print("Mean:", mean)
    print("Std :", std)


# ============================================================
# INTERPRETATION EXAMPLE
# ============================================================

print("\n🔍 Interpretation Example")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=SEED
)

perspectives = build_perspectives(X_train, y_train)
models = train_mpml(X_train, y_train, perspectives)

sample = X_test.iloc[[0]]

imp_scores = compute_impact_scores(models, sample)

print("\nPerspective Impact:")
for i, val in enumerate(imp_scores):
    print(f"P{i}: {val:.4f}")

top_idx = np.argmax(np.abs(imp_scores))
top_p = perspectives[top_idx]

feat_imp = compute_feature_impact(models, top_p, sample)

print("\nFeature Impact:")
for k, v in feat_imp.items():
    print(f"{k}: {v:.4f}")


# ============================================================
# CALIBRATION
# ============================================================

print("\n🔧 Calibration (Platt Scaling)")

cal = CalibratedClassifierCV(
    DecisionTreeClassifier(),
    method='sigmoid',
    cv=5
)

cal.fit(X_train, y_train)

print("Calibration Done.")

Dataset shape: (1000, 13)
Columns: ['patientid', 'age', 'gender', 'chestpain', 'restingBP', 'serumcholestrol', 'fastingbloodsugar', 'restingrelectro', 'maxheartrate', 'exerciseangia', 'oldpeak', 'slope', 'noofmajorvessels']

🚀 Running MPML...

MPML Results:
Accuracy Precision Recall F1
[0.977      0.97793396 0.98275862 0.98026692]
[0.01345362 0.01691678 0.01090441 0.01148441]

🚀 Running Baselines...

RandomForest
Mean: [0.978      0.98126529 0.98103448 0.98101664]
Std : [0.00748331 0.01177397 0.01432177 0.00651414]

Bagging
Mean: [0.976      0.98101773 0.97758621 0.97914098]
Std : [0.018      0.0121105  0.02563115 0.01594797]

Boosting
Mean: [0.955      0.97208562 0.95       0.96067734]
Std : [0.01204159 0.01345295 0.02241379 0.01092916]

🔍 Interpretation Example

Perspective Impact:
P0: 0.0314
P1: 0.0314
P2: 0.0314
P3: 0.0314
P4: -0.0520
P5: -0.0024
P6: -0.0520
P7: 0.0314
P8: -0.0071
P9: -0.0520
P10: -0.0163
P11: 0.0314
P12: -0.0065

Feature Impact:
patientid: -0.0769

🔧 Calibration (